<!-- nav -->
[← 8.08 · C Preprocessor](08_08_C_Preprocessor.ipynb) · [Home](../../README.md) · [8.10 · Cpp RegExp →](08_10_Cpp_RegExp.ipynb)

# 8.09 · Cpp Types

C++ writes types in more places than C: in a cast, and in `static_cast<int>(x)`, `new int`, `sizeof(int)`, a
conversion function's name, `operator int`, a lambda's trailing return type, a template parameter's default. What it
writes there is a **type-id** (ISO/IEC 14882, [dcl.name]), C's type name grown with C++'s own specifiers and
declarators: qualified names, `std::string`, references, `&`, scoped enumerations, `decltype`. `Cpp Types` models
type-ids as a dialect of its own, beside [8.07 · C Types](08_07_C_Types.ipynb), since a C++ type-id holds what no C type
name can. This page builds type-ids from `int` up, then an enumeration with a fixed type; then every place C++ puts one,
what validation says, how C++ reads one in a cast, and the editions it covers.

Like the rest of Part I, it starts from `dialects()`. `S` is its holder namespace and `amd64` the ABI a C family's
holder names (tutorial 2); `identifier` writes a C++ name of one identifier (tutorial 4), `x` the C++ term that reads
`x`, and `keyword` a type specifier that is a keyword, each a spec the builders call:

In [1]:
from mbse.Expressions.Cpp import Expressions as Cpp, Types as CppTypes
from mbse.Expressions.Framework import Standards, Symbolics, Terms
from toolkit import dialects

store = dialects()
S = store.Expressions
amd64 = store.Instances.Expressions.SystemV.AMD64
identifier = lambda word: lambda n: n.identifier(lambda i: i.spelling(word))
x = lambda t: t.idexpression(lambda r: r.name(identifier("x")))
keyword = lambda word: lambda s: s.primitivetypespecifier(lambda p: p.keyword(word))
DIALECT = CppTypes.Expressions.DIALECT


def refused(evaluate, expression):
    """What `evaluate` says of `expression`: its value, or the reason it refuses it."""
    try:
        return evaluate(expression)
    except (NotImplementedError, ValueError) as error:
        return str(error)

<a id="T08.09.1"></a>
## 08.09.1 · A type-id, from `int` to `const std::string&`

```text
int        unsigned long        const std::string&
```

`int` is a `typeid` of `Cpp Types`, as in `C Types`, its specifiers and declarator helpers; but they are C++'s helpers,
which C++'s own expressions hold too (a lambda's declarator, a `new` expression's), so `Cpp Types`' `typeid` is a
schema of its own, not C's. `const std::string&` shows two of them: a specifier whose name is a `qualifiedname`, `std`
then `string`, and a `referencedeclarator`. Its holder, `S.Cpp.Types.Expression`, names its ABI, as every C family
holder does:

In [2]:
int_ = lambda t: t.typeid(lambda i: i.specifiers([keyword("int")]))
with store.transaction():
    type_ids = [S.Cpp.Types.Expression(lambda e, spec=spec: e.root(spec).abi([amd64])) for spec in [
        int_,
        lambda t: t.typeid(lambda i: i.specifiers([keyword("unsigned"), keyword("long")])),
        lambda t: t.typeid(lambda i: i.specifiers([
            lambda s: s.cvqualifier(lambda q: q.keyword("const")),
            lambda s: s.namedtypespecifier(lambda n: n.name(lambda m: m.qualifiedname(
                lambda q: q.qualifiers([identifier("std")]).name(identifier("string")))))])
            .declarator(lambda d: d.referencedeclarator(lambda r: r)))]]
for type_id in type_ids:
    form = Terms.form(Terms.term(type_id))
    declarator = form.attributes.get("declarator")
    print(f"{CppTypes.Text.ToText(type_id):19}", form.kind, "of", Terms.dialect_of(Terms.term(type_id)), "| specifiers:",
          ", ".join(s.kind for s in form.attributes["specifiers"]), "| declarator:", declarator.kind if declarator else "-")
print(DIALECT.name(), DIALECT.tag, "| its kinds:", ", ".join(DIALECT.kinds()))
same = store.registered("Expressions.Cpp.Types.TypeId") is store.registered("Expressions.C.Types.TypeId")
print("its typeid the same schema as C Types':", "yes" if same else "no")
assert [CppTypes.Text.ToText(t) for t in type_ids] == ["int", "unsigned long", "const std::string&"]

int                 typeid of Expressions.Cpp.Types.Term | specifiers: primitivetypespecifier | declarator: -
unsigned long       typeid of Expressions.Cpp.Types.Term | specifiers: primitivetypespecifier, primitivetypespecifier | declarator: -
const std::string&  typeid of Expressions.Cpp.Types.Term | specifiers: cvqualifier, namedtypespecifier | declarator: referencedeclarator
Cpp Types cpp_types | its kinds: typeid, enumerator, error, missing
its typeid the same schema as C Types': no


<a id="T08.09.2"></a>
## 08.09.2 · A scoped enumeration, with its fixed type

```text
enum class Color : unsigned char {
    RED = 1,
    GREEN
}
```

C++'s enumerations may be scoped, `enum class`, and fix their underlying type, `: unsigned char`. The fixed type is a
type-id, a `Cpp Types` term the specifier holds as its `base`; each enumerator is `Cpp Types`' `enumerator`, its value
C++'s term. A form gives the specifier's terms in order, its base, then its enumerators:

In [3]:
enumerator = lambda word, value=None: lambda t: t.enumerator(
    lambda r: r.name(lambda i: i.spelling(word)) if value is None else r.name(lambda i: i.spelling(word)).value(value))
one = lambda t: t.literal(lambda r: r.value(lambda v: v.as_scalar(1)))
with store.transaction():
    color = S.Cpp.Types.Expression(lambda e: e.root(lambda u: u.typeid(lambda i: i.specifiers([
        lambda s: s.enumspecifier(lambda n: n.key("enum class").name(identifier("Color"))
            .base(lambda t: t.typeid(lambda i: i.specifiers([keyword("unsigned"), keyword("char")])))
            .body(True).enumerators([enumerator("RED", one), enumerator("GREEN")]))]))).abi([amd64]))
print(CppTypes.Text.ToText(color))
for t in Terms.form(Terms.term(color)).arguments:
    print(" ", f"{Terms.kind_of(t)[0]:10}", f"of {DIALECT.owner(t).name():9}", CppTypes.Text.ToText(t))
assert CppTypes.Text.ToText(color) == "enum class Color : unsigned char {\n    RED = 1,\n    GREEN\n}"

enum class Color : unsigned char {
    RED = 1,
    GREEN
}
  typeid     of Cpp Types unsigned char
  enumerator of Cpp Types RED = 1
  enumerator of Cpp Types GREEN


<a id="T08.09.3"></a>
## 08.09.3 · Wherever C++ puts a type

```text
(int)x        static_cast<int>(x)        new int        sizeof(int)        operator int        []<typename T = int> {}
```

C++'s kinds hold a `Cpp Types` term wherever its grammar puts a type-id, eight properties, read from the store as
tutorial 8 read them (08.3); and `typeid` stands among C++'s own kinds where an expression or a type may, in `sizeof`.
Each holder below is C++'s, and each prints the type where C++ writes it:

In [4]:
union = store.registered(DIALECT.union)
held = [f"{name}.{p}" for name in ("CastExpression", "NamedCastExpression", "NewExpression", "ConversionName",
                                   "LambdaDeclarator", "FunctionDeclarator", "EnumSpecifier", "TypeParameter")
        for p, prop in store.registered(f"Expressions.Cpp.Expressions.{name}").properties.items() if prop.schema is union]
print(", ".join(held))
with store.transaction():
    positions = [S.Cpp.Expressions.Expression(lambda e, spec=spec: e.root(spec).abi([amd64])) for spec in [
        lambda u: u.castexpression(lambda r: r.type(int_).operand(x)),
        lambda u: u.namedcastexpression(lambda n: n.operator("static_cast").type(int_).operand(x)),
        lambda u: u.newexpression(lambda n: n.type(int_)),
        lambda u: u.sizeofexpression(lambda s: s.operand(int_)),
        lambda u: u.idexpression(lambda r: r.name(lambda n: n.conversionname(lambda c: c.type(int_)))),
        lambda u: u.lambdaexpression(lambda r: r.template_parameters([lambda t: t.typeparameter(
            lambda p: p.key("typename").name(lambda i: i.spelling("T")).default(int_))]))]]
for expression in positions:
    print(f"{Cpp.Text.ToText(expression):24}", Terms.kind_of(Terms.term(expression))[0])
assert [Cpp.Text.ToText(p) for p in positions] == [
    "(int)x", "static_cast<int>(x)", "new int", "sizeof(int)", "operator int", "[]<typename T = int> {}"]

CastExpression.type, NamedCastExpression.type, NewExpression.type, ConversionName.type, LambdaDeclarator.trailing_return, FunctionDeclarator.trailing_return, EnumSpecifier.base, TypeParameter.default
(int)x                   castexpression
static_cast<int>(x)      namedcastexpression
new int                  newexpression
sizeof(int)              sizeofexpression
operator int             idexpression
[]<typename T = int> {}  lambdaexpression


<a id="T08.09.4"></a>
## 08.09.4 · Checked, and what a lambda binds in its type

Validation checks a type-id as C++'s grammar has it, and C++'s kinds check that a type stands where they need one: an
enumerator's name is an identifier, and a `new` expression needs a type. A type-id binds no name of its own, but it may
read one: `decltype(a)` reads `a`. In a lambda's trailing return type, `[](int a) -> decltype(a) {}`, `a` is the
lambda's parameter, which the lambda binds (tutorial 7), so the lambda needs nothing; without the parameter, `a` is
unbound, and validation says where:

In [5]:
decltype_a = lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.decltypespecifier(
    lambda d: d.expression(lambda e: e.idexpression(lambda r: r.name(identifier("a")))))]))
parameter = lambda p: p.parameterdeclaration(lambda q: q.specifiers([keyword("int")])
    .declarator(lambda e: e.iddeclarator(lambda i: i.name(identifier("a")))))
with store.transaction():
    checked = {
        "a b": S.Cpp.Types.Expression(lambda e: e.root(enumerator("a b")).abi([amd64])),
        "new": S.Cpp.Expressions.Expression(lambda e: e.root(lambda u: u.newexpression(lambda n: n)).abi([amd64])),
        "[](int a) -> decltype(a) {}": S.Cpp.Expressions.Expression(lambda e: e.root(lambda u: u.lambdaexpression(lambda r: r.declarator(
            lambda d: d.parameters([parameter]).trailing_return(decltype_a)))).abi([amd64])),
        "[] -> decltype(a) {}": S.Cpp.Expressions.Expression(lambda e: e.root(lambda u: u.lambdaexpression(lambda r: r.declarator(
            lambda d: d.trailing_return(decltype_a)))).abi([amd64])),
    }
for source, expression in checked.items():
    dialect = DIALECT if Terms.dialect_of(Terms.term(expression)) == DIALECT.union else Cpp.Expressions.DIALECT
    print(f"{source:27}", "|", "; ".join(dialect.validate(expression)) or "valid",
          "| needs:", ", ".join(sorted(Symbolics.free(expression, dialect))) or "nothing")
assert Cpp.Expressions.DIALECT.validate(checked["[](int a) -> decltype(a) {}"]) == []
assert Cpp.Text.ToText(checked["[](int a) -> decltype(a) {}"]) == "[](int a) -> decltype(a) {}"

a b                         | 'a b' is not an identifier | needs: nothing
new                         | a newexpression needs a type | needs: nothing
[](int a) -> decltype(a) {} | valid | needs: nothing
[] -> decltype(a) {}        | declarator trailing_return: specifiers 0 expression: idexpression 'a' is not bound | needs: a


<a id="T08.09.5"></a>
## 08.09.5 · Read by C++, in a cast

```text
(uint8_t)(x + 1)        (signed char)200        static_cast<int>(1)
```

C++ reads `Cpp Types`' terms through the dialects it reads (tutorial 8, 08.1), so it validates, infers and evaluates a
cast with its type in it, as C does ([8.07 · C Types](08_07_C_Types.ipynb), 08.07.5): `(uint8_t)(x + 1)` is `0` when `x` is
255; `(signed char)200` is -56, the ABI's `signed char` holding -128 to 127; and C++ infers the cast's domain from the
type it names. C++'s evaluator evaluates the kinds C++ shares with C, and refuses the others by name: a named cast,
`static_cast<int>(1)`, and a type-id itself:

In [6]:
uint8 = lambda t: t.typeid(lambda i: i.specifiers([lambda s: s.namedtypespecifier(lambda n: n.name(identifier("uint8_t")))]))
with store.transaction():
    casts = [S.Cpp.Expressions.Expression(lambda e, spec=spec: e.root(spec).abi([amd64])) for spec in [
        lambda u: u.castexpression(lambda r: r.type(uint8).operand(
            lambda o: o.binaryexpression(lambda b: b.left(x).operator("+").right(one)))),
        lambda u: u.castexpression(lambda r: r.type(lambda t: t.typeid(lambda i: i.specifiers([keyword("signed"), keyword("char")])))
            .operand(lambda o: o.literal(lambda l: l.value(lambda v: v.as_scalar(200))))),
        lambda u: u.namedcastexpression(lambda n: n.operator("static_cast").type(int_).operand(one))]]
for expression in casts:
    type_ = Terms.form(Terms.term(expression)).arguments[0]  # each one's first subterm, its type
    print(f"{Cpp.Text.ToText(expression):20}", "its type:", Terms.kind_of(type_)[0], "of",
          Cpp.Expressions.DIALECT.owner(type_).name(), "| x = 255:", refused(lambda c: Cpp.Simplifiers.OfAny(c, {"x": 255}).evaluate(), expression))
print("C++ infers", Cpp.Text.ToText(casts[1]), "as", Cpp.Expressions.DIALECT.infer(casts[1]).name())
print(refused(Cpp.Simplifiers.OfAny, type_ids[0]))
assert Cpp.Simplifiers.OfAny(casts[0], {"x": 255}).evaluate() == 0 and Cpp.Simplifiers.OfAny(casts[1]).evaluate() == -56

(uint8_t)(x + 1)     its type: typeid of Cpp Types | x = 255: 0
(signed char)200     its type: typeid of Cpp Types | x = 255: -56
static_cast<int>(1)  its type: typeid of Cpp Types | x = 255: Cpp namedcastexpression is not evaluated
C++ infers (signed char)200 as signed char
Cpp Types typeid is not evaluated


<a id="T08.09.6"></a>
## 08.09.6 · Its editions

`Cpp Types` covers C++'s three editions, as C++ does, each by its designation and its common name, both of its kinds in
each ([CAP-268: standards as tagged data](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-268.md)):

In [7]:
types = store.Instances.Expressions.Cpp.Types  # the dialect, by its global name
for designation, edition in Standards.standards(types).items():
    print(designation, "|", ", ".join(Standards.tags(edition)[1:]), "|", ", ".join(kind.name for kind in Standards.kinds(edition)))
assert Standards.find(types, "C++20") is Standards.standards(types)["ISO/IEC 14882:2020"]

ISO/IEC 14882:2017 | C++17 | Expressions.Cpp.Types.TypeId, Expressions.Cpp.Types.Enumerator
ISO/IEC 14882:2020 | C++20 | Expressions.Cpp.Types.TypeId, Expressions.Cpp.Types.Enumerator
ISO/IEC 14882:2024 | C++23 | Expressions.Cpp.Types.TypeId, Expressions.Cpp.Types.Enumerator


## What you learned

- **A type-id is a `Cpp Types` term**, `typeid`, of C++'s own specifiers and declarators: qualified names, references,
  `decltype`, scoped enumerations with a fixed type.
- **C++ holds one wherever its grammar puts a type**: a cast's, a named cast's, `new`'s, a conversion function's, a
  lambda's trailing return type, an enumeration's fixed type, a template parameter's default; and `sizeof(int)`.
- **It is checked as C++'s grammar has it**, and the names it reads are its scope's, a lambda's parameters among them.
- **C++ reads it in a cast**, converting and inferring by it, while a type-id itself is never evaluated.

Next, the regular expressions C++'s library reads, [8.10 · Cpp RegExp](08_10_Cpp_RegExp.ipynb).

<!-- nav -->
[← 8.08 · C Preprocessor](08_08_C_Preprocessor.ipynb) · [Home](../../README.md) · [8.10 · Cpp RegExp →](08_10_Cpp_RegExp.ipynb)